# Ninas and pangrams

Two things a setter can ask for on top of the style. Both constrain the fill
rather than the grid, and they work in opposite ways: a nina fixes letters
before the search begins, while a pangram leaves every letter free and changes
which words the search prefers.

1. What a nina is, and why it cannot be a fixed set of cells
2. Placing one, and checking it is actually hidden
3. How fixed letters enter the search
4. Pangrams: asking for every letter of the alphabet
5. What each one costs

In [1]:
import os
import sys
import time
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword import coverage, library
from crossword.fill import Filler
from crossword.index import Index
from crossword.render import show
from crossword.rules import RuleSet
from crossword.words import load

import make_grid

entries = load("crossword/UKACD.txt")
scores = {}
with open("crossword/scores.txt", encoding="utf-8") as handle:
    for line in handle:
        if not line.startswith("#") and line.strip():
            word, value = line.split()
            scores[word] = float(value)
index = Index(entries, scores)
patterns = library.load()

## 1. What a nina is

A message hidden in the grid, read somewhere the solver would not normally
read: around the perimeter, down a diagonal, along an unchecked column. The
answers themselves stay ordinary, and the message only appears to someone who
looks.

The name is borrowed from Al Hirschfeld, the American caricaturist, who from
1945 hid his daughter Nina's name in the lines of his drawings — in a sleeve, a
hairstyle, a curtain fold — and printed a number beside his signature saying
how many times. Crossword setters took the word for the same trick.

The awkward part is that a nina cannot be expressed as a fixed set of cells.
The perimeter of a 15x15 grid is 56 squares, but a grid has blocks on its edges
and the message runs through whatever white squares remain.

In [2]:
perimeter = make_grid.PATHS["perimeter"](15)
print(f"the perimeter of a 15x15 is {len(perimeter)} cells\n")

white = Counter()
for pattern in patterns:
    white[sum(1 for cell in perimeter if cell not in pattern.blocks)] += 1

print("white cells on the perimeter, across the 120 published grids:")
for count, grids in sorted(white.items(), reverse=True)[:6]:
    print(f"  {count:2d} white   {grids:3d} grids")
print(f"\nall 56 open: {white.get(56, 0)} grids")

the perimeter of a 15x15 is 56 cells

white cells on the perimeter, across the 120 published grids:
  56 white     3 grids
  54 white     5 grids
  52 white    34 grids
  50 white    10 grids
  48 white     8 grids
  46 white     2 grids

all 56 open: 3 grids


So the message is laid *relative to the grid*: take the white cells along the
path in order, and let the blocks interrupt it. A setter reads a perimeter nina
by going round and skipping the black squares, and that is exactly how it has
to be placed.

The paths available are the ones a message is worth hiding along.

In [3]:
for name in sorted(make_grid.PATHS):
    cells_on_path = make_grid.PATHS[name](15)
    print(f"  {name:14} {len(cells_on_path):2d} cells")

  antidiagonal   15 cells
  bottomrow      15 cells
  diagonal       15 cells
  leftcol        15 cells
  perimeter      56 cells
  rightcol       15 cells
  toprow         15 cells


## 2. Placing one

`nina_placer` turns a path and a message into a rule the pattern search can
apply: for a given grid, which cell holds which letter. A grid with too few
white cells on the path cannot carry the message and is rejected before any
filling is attempted.

A perimeter nina takes an extra condition. The message must use *every* white
cell on the path, because a circuit that stops three quarters of the way round
is not a perimeter nina. An open path like a diagonal has no such requirement,
and a short message simply sits at the start of it.

In [4]:
path, text = make_grid.read_nina_path("diagonal,IN PLAIN SIGHT")

# `exact` makes the message occupy the whole path rather than starting at one
# end and stopping where it runs out. A perimeter always needs it -- a circuit
# that halts three quarters of the way round is not one -- and on a diagonal it
# is the difference between a message *down* the diagonal and a message that
# *is* the diagonal.
place = make_grid.nina_placer(path, text, exact=True)
print(f"message {text.upper()!r} ({len(text)} letters) filling the diagonal")

carriers = [p for p in patterns if place(p) is not None]
loose = [p for p in patterns
         if make_grid.nina_placer(path, text)(p) is not None]
print(f"{len(loose)} of {len(patterns)} grids have room for it")
print(f"{len(carriers)} have exactly {len(text)} white cells on the diagonal")

for cell, letter in list(place(carriers[0]).items())[:4]:
    print(f"  {cell} <- {letter.upper()}")
print("  ...")

message 'INPLAINSIGHT' (12 letters) filling the diagonal
29 of 120 grids have room for it
6 have exactly 12 white cells on the diagonal
  (1, 1) <- I
  (2, 2) <- N
  (3, 3) <- P
  (4, 4) <- L
  ...


In [5]:
began = time.time()
result = coverage.best_over_library(
    carriers, index, [], RuleSet(), time_limit=90,
    commonness=3.0, aim=0.85, seed=2, preset=place,
)
print(f"{time.time() - began:.1f}s, complete: {result.ok}")
grid = result.grid

# Which cells carry the message depends on where *this* grid's blocks fall, so
# it has to be recomputed for the pattern that was chosen.
message_cells = set(place(result.pattern))
show(grid, highlight=message_cells)

0.1s, complete: True


## 3. Is it hidden?

A message is only a nina if reading it tells the solver something the answers
do not. If the cells carrying it are exactly a set of whole entries, then
reading them just reads those answers: MYSTERY and THEATRE along a top row
that happens to be two seven-letter entries is not a nina, it is 1 and 5
Across.

So the test is whether any of the message's letters fall outside the entries
that lie wholly inside it.

In [6]:
whole = [s for s in grid.slots() if set(s.cells) <= message_cells]
covered = {c for s in whole for c in s.cells}
exposed = message_cells - covered
touched = {(s.direction, s.row, s.col) for s in grid.slots()
           for c in message_cells if c in s.cells}

print(f"the message spans {len(touched)} entries")
print(f"entries lying wholly inside it: {len(whole)}")
print(f"letters not readable as a whole entry: "
      f"{len(exposed)} of {len(message_cells)}")
print("\nhidden" if exposed else "\nNOT hidden")

reading = "".join(grid.letters[c] for c in path if c in message_cells)
print(f"\nread down the diagonal: {reading.upper()}")

the message spans 12 entries
entries lying wholly inside it: 0
letters not readable as a whole entry: 12 of 12

hidden

read down the diagonal: INPLAINSIGHT


## 4. How fixed letters enter the search

They are simply written into the grid before the search starts, and everything
downstream honours them without being told. A slot's pattern is read from the
grid, so an entry crossing a nina letter already has that letter in its
pattern, and the index will only ever offer words matching it.

One thing does have to be checked afterwards. Where the path runs *along* an
entry rather than across it, the nina can fix that entry completely — a
perimeter message lying on the top row fills whichever entries make up that
row. Those entries are then not chosen by the search at all; they are whatever
letters the message happened to put there, and there is no reason for them to
be words.

So the message has to divide into real words at the points where one entry ends
and the next begins. If the top row is a seven and a seven, the first seven
letters and the last seven each have to be a word in their own right. When they
are not, the grid still passes every rule — the letters are all there and the
crossings all agree — so the builder says which entries came out as nonsense
rather than reporting it clean.

In [7]:
crossing = [s for s in grid.slots() if set(s.cells) & message_cells]
print("entries the message passes through, and what they became:")
for slot in crossing[:8]:
    letters = sum(1 for c in slot.cells if c in message_cells)
    print(f"  {grid.pattern(slot).upper():16} "
          f"{letters} of its {slot.length} letters fixed by the nina")

entries the message passes through, and what they became:
  MNEMONICS        1 of its 9 letters fixed by the nina
  REELING          1 of its 7 letters fixed by the nina
  ASTERISKED       1 of its 10 letters fixed by the nina
  LEITMOTIVS       1 of its 10 letters fixed by the nina
  LABIALS          1 of its 7 letters fixed by the nina
  APPLEJOHN        1 of its 9 letters fixed by the nina
  IMPROBABILITIES  1 of its 15 letters fixed by the nina
  REPELLED         1 of its 8 letters fixed by the nina


## 5. Pangrams

A pangram uses every letter of the alphabet at least once; `--pangram 2` asks
for each of them twice.

Nothing is fixed in advance here. Instead the search keeps a mask of the
letters the grid still lacks, and words supplying a missing letter are pulled
forward in the candidate order. The bonus is *per missing letter*, so a word
supplying both a Q and a Z outranks one supplying either — and once a letter is
in the grid its pull vanishes, which is what stops the fill flooding with
awkward words once the rare letters are covered.

In [8]:
def fill_with(pangram, seed=5):
    made = patterns[0].grid()
    worker = Filler(made, index, RuleSet(), seed=seed, commonness=3.0,
                    aim=0.85, pangram=pangram, node_budget=60000)
    began = time.time()
    if not worker.fill():
        return None, None, None
    letters = Counter("".join(made.letters.values()))
    ranks = []
    for slot in made.slots():
        bucket = index.lengths[slot.length]
        word_id = bucket.by_word.get(made.pattern(slot))
        if word_id is not None and bucket.quantile:
            ranks.append(bucket.quantile[word_id])
    return made, letters, (sum(ranks) / len(ranks), time.time() - began)


for want in (0, 1):
    made, letters, stats = fill_with(want)
    missing = [c for c in "abcdefghijklmnopqrstuvwxyz" if letters[c] < 1]
    rare = " ".join(f"{c}:{letters[c]}" for c in "jqxzkvw")
    label = "no requirement" if want == 0 else f"pangram x{want}"
    print(f"{label:16} missing {len(missing):2d} letters "
          f"({''.join(missing).upper() or 'none'})")
    print(f"{'':16} rare letters  {rare}")
    print(f"{'':16} familiarity {stats[0]:.2f}\n")

no requirement   missing  5 letters (JKQXZ)
                 rare letters  j:0 q:0 x:0 z:0 k:0 v:4 w:3
                 familiarity 0.79

pangram x1       missing  0 letters (none)
                 rare letters  j:1 q:1 x:1 z:1 k:4 v:1 w:1
                 familiarity 0.79



In [9]:
import statistics

print("request        filled   familiarity   time")
for want in (0, 1, 2, 3):
    ranks, times, done = [], [], 0
    for seed in range(8):
        made, letters, stats = fill_with(want, seed=seed)
        if made is None:
            continue
        done += 1
        ranks.append(stats[0])
        times.append(stats[1])
    label = "none" if want == 0 else f"pangram x{want}"
    if ranks:
        print(f"  {label:12}    {done}/8       {statistics.mean(ranks):.2f}"
              f"       {statistics.mean(times):5.1f}s")
    else:
        print(f"  {label:12}    0/8       --")

request        filled   familiarity   time
  none            8/8       0.79         0.0s
  pangram x1      8/8       0.75         0.0s
  pangram x2      8/8       0.69         0.0s
  pangram x3      6/8       0.63         0.2s


Completions hold up well and the cost appears in the familiarity instead,
falling steadily as more of the grid is dictated. Even a triple mostly comes
out, which is worth remarking on: it asks for 78 of the roughly 160 letters in
a British grid, most of them from the alphabet's awkward end.

Two things make that possible. The pull towards a letter lasts only until that
letter has been placed often enough, so it does not go on flooding the grid
with awkward words once the requirement is met. And the pull is weighted by how
hard the letter is to place, so the Q and the J go in while the grid is still
mostly empty and there is somewhere to put them. Weighting every missing letter
equally instead puts the easy ones in first and spends exactly the freedom the
hard ones needed: under that scheme a double pangram completes 7 times in 12
and a triple never.

In [10]:
made, letters, stats = fill_with(1)
show(made)

## 6. What they cost

Both constraints are paid for in the fill, and the currency is familiarity: the
more of the grid a setter dictates, the less freedom the search has to choose
ordinary words for the rest. Over eight seeds on one grid:

A nina costs in a different way. It does not change which words are preferred;
it removes grids from consideration — only the patterns whose path has room for
the message are even tried — and it fixes letters that every crossing entry
must then accommodate.

## Where this ends

That is the whole builder: a dictionary and an index over it, a library of
grids, a rule set per style, a search that seats chosen words and fills the
rest, and these two constraints on top.

- [1. Building a grid](01-building-a-grid.ipynb)
- [2. Words and the index](02-words-and-the-index.ipynb)
- [3. The fill search](03-the-fill-search.ipynb)
- [4. The other styles](04-the-other-styles.ipynb)